# Axial porosity distribution comparison

Compare local porosity distributions between real and generated digital cores over axial layer ranges. The example uses the Z direction and 128³ int8 binary masks with 1=pore and 0=solid, with TI, DCGAN, STGAN, StyleGAN and ours groups.


In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np

try:
    from scipy.interpolate import make_interp_spline
except ImportError:
    make_interp_spline = None

# ==========================================
# 1. Parameters and data paths
# ==========================================
# Interpret RAW arrays as (Z, Y, X)
DIMENSION = (128, 128, 128)
DATA_TYPE = np.int8
PORE_VALUE = 1

FOLDERS = {
    "TI": r"../../data/real_binary",
    "DCGAN": r"../../data/baselines/dcgan",
    "StyleGAN": r"../../data/baselines/stylegan",
    "STGAN": r"../../data/baselines/stgan",
    "ours": r"../../data/generated_binary",
}

STYLES = {
    "TI": {"color": "#C00000", "marker": "o", "label": "TI"},
    "DCGAN": {"color": "#2F5597", "marker": "^", "label": "DCGAN"},
    "STGAN": {"color": "#595959", "marker": "d", "label": "STGAN"},
    "StyleGAN": {"color": "#8FAADC", "marker": "s", "label": "StyleGAN"},
    "ours": {"color": "#A6A6A6", "marker": "*", "label": "ours"},
}

MAX_FILES_PER_GROUP = None  # None includes all available files in each group.
SMOOTH_POINTS = 500
MARK_EVERY = 8
SHOW_STD_BAND = True

# Select the slice axis: 0=Z, 1=Y, 2=X. The default computes Z-slice profiles.
LAYER_AXIS = 0
LAYER_RANGES = [(0, 32), (32, 64), (64, 96), (96, 128)]

print("Configured folders:")
for name, folder in FOLDERS.items():
    print(f"  {name}: {folder}")

In [ ]:
# ==========================================
# 2. Load data and calculate slice porosity profiles
# ==========================================
def list_raw_files(folder, max_files=None):
    files = sorted(glob.glob(os.path.join(folder, "*.raw")))
    if max_files is not None:
        files = files[:max_files]
    return files


def load_binary_rock(filepath):
    raw = np.fromfile(filepath, dtype=DATA_TYPE)
    expected = int(np.prod(DIMENSION))
    if raw.size != expected:
        raise ValueError(
            f"{filepath} has {raw.size} voxels, expected {expected}. "
            f"Check DIMENSION or DATA_TYPE."
        )
    volume = raw.reshape(DIMENSION)
    return volume == PORE_VALUE


def calc_porosity_sequence(filepath, layer_axis=0):
    """Calculate porosity sequence along one axis.

    For DIMENSION=(Z,Y,X), layer_axis=0 gives true Z-layer porosity.
    """
    is_pore = load_binary_rock(filepath)
    mean_axes = tuple(axis for axis in range(3) if axis != layer_axis)
    return is_pore.astype(np.float32).mean(axis=mean_axes)


def get_sequences_for_folder(folder_path):
    files = list_raw_files(folder_path, MAX_FILES_PER_GROUP)
    if not files:
        print(f"Warning: no .raw files found in {folder_path}")
        return None

    sequences = []
    for file_path in files:
        sequences.append(calc_porosity_sequence(file_path, layer_axis=LAYER_AXIS))
    return np.asarray(sequences, dtype=np.float64)


results = {}
axis_names = {0: "Z", 1: "Y", 2: "X"}
print(f"Start extracting porosity sequences along {axis_names[LAYER_AXIS]} axis...")

for name, folder in FOLDERS.items():
    print(f"\nProcessing {name}...")
    sequences = get_sequences_for_folder(folder)
    if sequences is None:
        continue
    sample_porosity = sequences.mean(axis=1)
    results[name] = {
        "mean": sequences.mean(axis=0),
        "std": sequences.std(axis=0, ddof=1) if len(sequences) > 1 else np.zeros(sequences.shape[1]),
        "sample_porosity": sample_porosity,
        "porosity_mean": float(sample_porosity.mean()),
        "porosity_std": float(sample_porosity.std(ddof=1)) if len(sample_porosity) > 1 else 0.0,
        "n": len(sequences),
    }
    print(f"  samples: {len(sequences)}, average porosity: {sequences.mean():.6f}")

if not results:
    raise RuntimeError("No data available. Please check folder paths.")

In [ ]:
# ==========================================
# 3. Compare mean porosity over the selected layer ranges
# ==========================================
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"] + plt.rcParams["font.serif"]

names = [name for name in FOLDERS.keys() if name in results]
range_labels = [f"{start}-{end}" for start, end in LAYER_RANGES]

# segment_means[range_idx, model_idx]
segment_means = np.zeros((len(LAYER_RANGES), len(names)), dtype=float)

for j, name in enumerate(names):
    mean_seq = results[name]["mean"]
    for i, (start, end) in enumerate(LAYER_RANGES):
        start = max(0, int(start))
        end = min(len(mean_seq), int(end))
        if end <= start:
            raise ValueError(f"Invalid layer range: {(start, end)}")
        segment_means[i, j] = mean_seq[start:end].mean()

fig, ax = plt.subplots(figsize=(9.0, 5.0), dpi=150)
x = np.arange(len(LAYER_RANGES))
bar_width = min(0.78 / max(len(names), 1), 0.16)
offsets = (np.arange(len(names)) - (len(names) - 1) / 2) * bar_width

for j, name in enumerate(names):
    style = STYLES[name]
    ax.bar(
        x + offsets[j],
        segment_means[:, j],
        width=bar_width,
        color=style["color"],
        edgecolor="black",
        linewidth=0.8,
        alpha=0.88,
        label=style["label"],
        zorder=3,
    )


# Compare each method with the TI reference profile
if "TI" in names:
    ti_idx = names.index("TI")
    ref_color = "#f28e2b"
    group_half_width = (len(names) * bar_width) / 2
    for i in range(len(LAYER_RANGES)):
        ti_value = segment_means[i, ti_idx]
        left = x[i] - group_half_width - bar_width * 0.15
        right = x[i] + group_half_width + bar_width * 0.15
        ax.hlines(
            ti_value,
            left,
            right,
            colors=ref_color,
            linestyles="--",
            linewidth=1.0,
            zorder=6,
        )

        for j, name in enumerate(names):
            value = segment_means[i, j]
            if value >= ti_value:
                continue
            bar_center = x[i] + offsets[j]
            hatch_color = STYLES[name]["color"]

            # Fill the gap to the TI reference line using the bar color.
            ax.bar(
                bar_center,
                ti_value - value,
                width=bar_width,
                bottom=value,
                facecolor="none",
                edgecolor=hatch_color,
                linewidth=0.0,
                hatch="////",
                alpha=0.65,
                zorder=1,
            )

            # Outline the filled region in black.
            ax.bar(
                bar_center,
                ti_value - value,
                width=bar_width,
                bottom=value,
                facecolor="none",
                edgecolor="black",
                linewidth=0.8,
                linestyle="-",
                zorder=2,
            )

            # Redraw the black outline so the colored fill does not obscure the bar border.
            ax.bar(
                bar_center,
                value,
                width=bar_width,
                bottom=0,
                facecolor="none",
                edgecolor="black",
                linewidth=0.8,
                linestyle="-",
                zorder=2,
            )

ax.set_xticks(x)
ax.set_xticklabels(range_labels, fontsize=11)
ax.set_xlabel(f"Layer Range ({axis_names[LAYER_AXIS]}-axis)", fontsize=13, labelpad=8)
ax.set_ylabel("Porosity", fontsize=13, labelpad=8)
ax.set_title("Axial porosity distribution comparison", fontsize=14, pad=8)

upper = float(np.max(segment_means)) if segment_means.size else 1.0
lower = float(np.min(segment_means)) if segment_means.size else 0.0
pad = max((upper - lower) * 0.30, 0.02)
ax.set_ylim(max(0.0, lower - pad), min(1.0, upper + pad))

ax.grid(axis="y", linestyle="--", alpha=0.25, zorder=0)
ax.tick_params(axis="both", which="major", direction="in", length=6, labelsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(loc="best", frameon=True, fontsize=10, edgecolor="black", fancybox=False, ncol=2)

# Export the mean porosity for each layer range and method
header_cols = ["layer_range"] + [f"{name}_mean" for name in names]
rows = []
for i, label in enumerate(range_labels):
    row = [label] + [f"{segment_means[i, j]:.8f}" for j in range(len(names))]
    rows.append(row)
with open("porosity_layer_range_bar_values.csv", "w", encoding="utf-8") as f:
    f.write(",".join(header_cols) + "\n")
    for row in rows:
        f.write(",".join(row) + "\n")

plt.tight_layout()
plt.savefig("porosity_layer_range_bar_chart.png", dpi=600, bbox_inches="tight")
plt.savefig("porosity_layer_range_bar_chart.pdf", bbox_inches="tight")
plt.show()

print("Saved figures: porosity_layer_range_bar_chart.png / porosity_layer_range_bar_chart.pdf")
print("Saved data: porosity_layer_range_bar_values.csv")
